In [4]:
# -*- coding: UTF-8 -*-
import warnings
from pyspark.sql import SparkSession
from pyspark.sql.types import (IntegerType, StringType, DateType, DecimalType, TimestampType)
from pyspark.sql.functions import row_number, col, when, trim, lit, current_timestamp, max as spark_max, to_date
from pyspark.sql.window import Window  
warnings.filterwarnings("ignore")
from datetime import datetime, timedelta
import calendar

# 仅保留有效业务字段（删除不存在的dlv_rate/updatedate)
TARGET_FIELD_ORDER = [
    "id",  # ID字段（后续分配）
    # 核心业务字段
    "code", "shift_date", "shift_no", "bp_code", "bp_name",
    "is_bp_station", "bp_station_name", "vehicle_type_code", "vehicle_type",
    "license_plate", "all_weight", "all_weight_kg", "cane_weight", "cane_weight_kg",
    "vehicle_weight", "vehicle_weight_kg", "trash_weight", "trash_weight2", "trash_perc",
    # 质检字段
    "first_check_cane_tail", "first_check_cane_leaf", "first_check_root",
    "first_check_sediment", "first_check_pest", "first_check_impurity",
    "first_insp_reject_blackhead","first_insp_reject_core_clump","over_ton_impurity_ded",
    "second_check_cane_tail", "second_check_cane_leaf", "second_check_root",
    "second_check_sediment", "second_check_pest", "second_check_impurity",
    "second_insp_reject_blackhead","second_insp_reject_core_clump",
    "first_check_inspector", "second_check_inspector",
    "pre_discount_imp_cane_qty", "post_discount_imp_cane_qty",
    # 时间/状态字段
    "trash_deduct", "arrival_time", "departure_time",
    "all_weight_timestamp", "vehicle_weight_timestamp", "weight_type",
    # 物流相关
    "dump_seq_no", "dump_aux_code", "dump_number", "tray_code",
    "dump_timestamp", "dump_timestamp2", "queue_code",
    # 单据相关
    "receipt_no", "receipt_no_series",
    # 质检数值
    "ccs", "raw_ccs", "raw_ccs2", "purity", "purity2",
    "brix", "brix2", "pol", "pol2", "fiber", "fiber2",
    # 配置相关
    "loading_method","loading_method_detail", "cane_grade_code", "cane_grade_group",
    "cane_grade_group_name", "cane_grade_name",
    # 企业/站点信息
    "company_code", "company_name", "company", "station_code", "station_type",
    "vehicle_code", "trailer_truck",
    "cane_recipient_company_code", "zone_code", "zone_name",
    "vehicle_bp_code", "vehicle_bp_name",
    # 时间/地域信息
    "cut_timestamp", "be_year",
    "town_code", "town_name", "village_code", "village_name",
    "team_code", "team_name",
    # ========== 新增：添加harvest_team和harvest_vehicle字段 ==========
    "harvest_team", "harvest_vehicle",
    # ========== 新增结束 ==========
    "variety_code", "variety_name",
    "variety_type_code", "variety_type_name",
    # 系统字段
    "create_time", "update_time", "cqms_create_time", "cqms_update_time",
    "large_vehicle_receipt", "sucrose_cane", "gpj", "rs", "bc", "pc",
    "swipe_card", "press_line","is_traceqc",
    "dw_update_time"
]

def generate_hive_ddl(df, table_name, database='default', 
                     partitioned_columns=None, field_name_mapping=None):
    """生成简化版Hive分桶表DDL（仅保留核心：字段+分区+分桶）"""
    type_mapping = {
        'StringType': 'STRING',
        'IntegerType': 'INT',
        'LongType': 'BIGINT',
        'DoubleType': 'DOUBLE',
        'FloatType': 'FLOAT',
        'BooleanType': 'BOOLEAN',
        'DateType': 'DATE',
        'TimestampType': 'TIMESTAMP',
        'DecimalType': lambda t: f"DECIMAL({t.precision},{t.scale})",
        'ArrayType': lambda t: f"ARRAY<{_parse_type(t.elementType)}>",
        'MapType': lambda t: f"MAP<{_parse_type(t.keyType)},{_parse_type(t.valueType)}>",
        'StructType': lambda t: f"STRUCT<{','.join([f'{field.name}:{_parse_type(field.dataType)}' for field in t.fields])}>"
    }

    def _parse_type(data_type):
        """递归解析嵌套类型"""
        type_name = data_type.__class__.__name__
        if type_name == 'DecimalType':
            return type_mapping[type_name](data_type)
        elif type_name in ['ArrayType', 'MapType', 'StructType']:
            return type_mapping[type_name](data_type)
        else:
            return type_mapping.get(type_name, 'STRING')

    def to_pinyin_field(origin_name):
        if field_name_mapping:
            return field_name_mapping.get(origin_name, origin_name)
        return origin_name

    # 1. 构建字段列表
    columns = []
    partitioned_columns = partitioned_columns or []
    for field in df.schema.fields:
        if field.name in partitioned_columns:
            continue
        field_name_pinyin = to_pinyin_field(field.name)
        col_type = _parse_type(field.dataType)
        comment = f"COMMENT '{field.name}'"
        columns.append(f"  `{field_name_pinyin}` {col_type} {comment}".strip())
    columns_str = ', \n'.join(columns)

    # 2. 构建分区字段
    partitioned_str = ''
    if partitioned_columns:
        partition_cols = [
            f"`{col}` string COMMENT 'DataSail Generate Partition Column: {col}'" 
            for col in partitioned_columns
        ]
        partitioned_str = "\nPARTITIONED BY ( \n  " + \
                          ", \n  ".join(partition_cols) + \
                          "\n)"

    # 3. 构建分桶语句（仅保留核心分桶定义）
    bucketed_str = """
CLUSTERED BY ( 
  id) 
INTO 32 BUCKETS"""

    # 4. 拼接简化版完整DDL（移除存储格式、LOCATION、TBLPROPERTIES）
    ddl = f"""CREATE TABLE `{database}`.`{table_name}`(
  {columns_str}
){partitioned_str}{bucketed_str}"""
    
    # 格式化缩进
    ddl = ddl.replace('  ', '  ').strip()
    return ddl

def write_to_partition(spark, df, database, table_name, dt, dh, dm, table_exists_flag, is_overwrite=False):
    """通用分区写入（处理分桶表创建/分区管理）"""
    # 1. 首次建表：按id分32桶（简化版DDL）
    if not table_exists_flag:
        print(f"+++++++++表{database}.{table_name}不存在, 创建简化版分桶表（id/32桶）++++++++++++")
        ddl = generate_hive_ddl(
            df=df,
            table_name=table_name,
            database=database,
            partitioned_columns=["dt", "dh", "dm"]
        )
        print(f"建表DDL：\n{ddl}")
        spark.sql(ddl)
    
    # 2. 检查/创建分区
    try:
        partitions = spark.sql(f"SHOW PARTITIONS {database}.{table_name}").collect()
        target_partition = f"dt={dt}/dh={dh}/dm={dm}"
        partition_exists = any(target_partition in str(p) for p in partitions)
    except:
        partition_exists = False
    
    if not partition_exists:
        print(f"+++++++++新建分区：{dt}/{dh}/{dm}++++++++++++")
        spark.sql(f"ALTER TABLE {database}.{table_name} ADD PARTITION (dt='{dt}', dh='{dh}', dm='{dm}')")
    
    # 3. 写入数据（分桶写入）
    # ========== 核心修改：分桶表强制使用OVERWRITE模式 ==========
    # Hive分桶表不支持INSERT INTO，必须用INSERT OVERWRITE（即使是新分区）
    write_mode = "OVERWRITE"
    temp_view_name = f"temp_view_{dt}_{dh}_{dm}"
    df.createOrReplaceTempView(temp_view_name)
    
    # 仅获取有效字段
    table_cols = spark.sql(f"DESCRIBE {database}.{table_name}").select("col_name").collect()
    valid_cols = [
        row["col_name"] for row in table_cols 
        if row["col_name"] and not row["col_name"].startswith("#") and row["col_name"] not in ["dt", "dh", "dm"]
    ]
    col_list_str = ", ".join(valid_cols)
    
    # 分桶写入
    spark.sql(f"""
        INSERT {write_mode} {database}.{table_name} PARTITION (dt='{dt}', dh='{dh}', dm='{dm}') ({col_list_str})
        SELECT {col_list_str} FROM {temp_view_name} CLUSTER BY id 
    """)

    spark.sql(f"DROP TABLE IF EXISTS {temp_view_name}")
    print(f"+++++++++分区{dt}/{dh}/{dm}数据写入完成++++++++++++")

def writer_las(spark, df, database, table_name, dt, dh, dm, is_overwrite=False):
    """主写入函数（修复：添加spark作为参数，避免作用域错误）"""
    spark.sql(f"use {database};")
    # 检查表是否存在
    try:
        table_df = spark.sql("show tables;").toPandas()
        table_exists_flag = table_name in table_df['tableName'].values
    except Exception as e:
        print(f"检查表是否存在失败：{e}")
        table_exists_flag = False
    # 调用写入分区函数
    write_to_partition(
        spark=spark,
        df=df,
        database=database,
        table_name=table_name,
        dt=dt,
        dh=dh,
        dm=dm,
        table_exists_flag=table_exists_flag,
        is_overwrite=is_overwrite
    )

def get_month_date_range(month_str):
    """
    根据月份字符串（yyyyMM）计算查询时间范围：上月最后3天 ~ 当月最后1天
    返回：(start_date, end_date) 格式为 'YYYY-MM-DD'
    """
    # 解析月份
    year = int(month_str[:4])
    month = int(month_str[4:])
    
    # 当月第一天和最后一天
    first_day = datetime(year, month, 1)
    last_day = datetime(year, month, calendar.monthrange(year, month)[1])
    
    # 上月最后3天的起始日期
    last_month_last_day = first_day - timedelta(days=1)
    start_date = last_month_last_day - timedelta(days=2)
    
    # 格式化日期字符串
    return start_date.strftime('%Y-%m-%d'), last_day.strftime('%Y-%m-%d')

def get_factory_data_by_month(spark, target_month=None, is_history=False, no_date_filter=False):
    """
    按月份/全量获取工厂数据（核心修改：
    1. 新增no_date_filter参数，控制是否去掉日期筛选
    2. is_history=True → a.season != '25/26'（历史数据）
    3. is_history=False → a.season = '25/26'（2526榨季）
    """
    # ========== 批量刷新所有用到的表 ==========
    tables_to_refresh = [
        "dwd.dwd_gb_settle_info_i_1h",
        "dwd.dwd_gis_sugarcane_quality_tracking_i_1h",
        "dwd.dwd_gis_team_info_i_1mon",
        "dwd.dwd_gis_village_info_i_1mon",
        "dwd.dwd_gis_town_info_f_1mon",
        "dwd.dwd_gis_sugarcane_owner_info_i_1mon",
        "dwd.dwd_gis_nw_company_info_i_1mon",
        "dwd.dwd_gis_dispatch_plan_info",
        "dwd.dwd_gis_transfer_station_info_i_1mon",
        "dwd.dwd_gis_vehicle_model_info_i_1mon",
        "dwd.dwd_gis_fleet_info_i_1mon",
        "dwd.dwd_gis_vehicle_dispatch_info",
        "dwd.dwd_gis_second_quality_inspection_f_1h",
        "dwd.dwd_cqms_zc_cph_zzh_swipe_info_i_1h",
        "dwd.dwd_gis_sugarcane_zone_team_i_1mon",
        "dwd.dwd_gis_ticket_category_info_i_1mon",
        "dwd.dwd_gis_sugarcane_variety_info_i_1mon",
        "dwd.dwd_gis_variety_price_category_f_1mon"
    ]
    
    # 循环刷新所有表（根据是否有月份调整打印信息）
    if no_date_filter:
        print(f"\n======= 开始刷新依赖表（全量历史数据） =======")
    else:
        print(f"\n======= 开始刷新{target_month}月依赖表 =======")
    
    for table in tables_to_refresh:
        try:
            spark.sql(f"REFRESH TABLE {table}")
            print(f"✅ 成功刷新表：{table}")
        except Exception as e:
            print(f"❌ 刷新表{table}失败：{str(e)}")
    
    if no_date_filter:
        print(f"======= 依赖表刷新完成（全量历史数据） =======")
    else:
        print(f"======= {target_month}月依赖表刷新完成 =======")
    
    # ========== 计算时间范围：仅非全量历史数据时生效 ==========
    date_cond = ""
    if not no_date_filter and target_month:
        start_date, end_date = get_month_date_range(target_month)
        print(f"\n======= {target_month}月数据筛选范围 =======")
        print(f"1. pzs日期范围：{start_date} 至 {end_date}（含上月最后3天缓冲区）")
        date_cond = f"AND TO_DATE(a.pzs) >= '{start_date}' AND TO_DATE(a.pzs) <= '{end_date}'"
    elif no_date_filter:
        print(f"\n======= 数据筛选范围 =======")
        print(f"1. 无日期范围限制（全量历史数据）")
    
    # ========== 核心修改：替换为season字段筛选条件 ==========
    if is_history:
        season_cond = "AND a.season != '25/26'"
        print(f"2. season筛选条件：{season_cond.strip('AND ')}（历史数据）")
    else:
        season_cond = "AND a.season = '25/26'"
        print(f"2. season筛选条件：{season_cond.strip('AND ')}（2526榨季数据）")
    
    # ========== 构建筛选SQL ==========
    sql = f"""
SELECT  
DISTINCT
    a0.pyjiancheng AS code,
    CAST(pzs AS DATE) AS shift_date,  
    CASE 
        WHEN HOUR(TO_TIMESTAMP(pzs, 'yyyy-MM-dd HH:mm:ss')) BETWEEN 0 AND 7 THEN 1
        WHEN HOUR(TO_TIMESTAMP(pzs, 'yyyy-MM-dd HH:mm:ss')) BETWEEN 8 AND 15 THEN 2
        ELSE 3 
    END AS shift_no,
    a.zzm AS bp_code,
    a14.zzmc AS bp_name,
    NVL(aj6.zzzt, '0') AS is_bp_station,
    zzz.zzzmc AS bp_station_name,    
    vtype.fl AS vehicle_type_code,
    vtype.cmc AS vehicle_type,
    a.gaph AS license_plate,
    a.mz * 0.001 AS all_weight,
    a.mz AS all_weight_kg,
    a.jz * 0.001 AS cane_weight,
    a.jz AS cane_weight_kg,
    a.pp * 0.001 AS vehicle_weight,
    a.pp AS vehicle_weight_kg,
    ROUND((a.mz - a.pp) * a.kz * 0.01 * 0.001, 3) AS trash_weight,
    ROUND((a.mz - a.pp) * a.kc * 0.01 * 0.001, 3) AS trash_weight2,
    a.kz  AS trash_perc,

    -- 质检字段
    a.gbkzm as first_check_cane_tail,
    a.gbkyq as first_check_cane_leaf,
    a.gbkxg as first_check_root,
    a.gbkns as first_check_sediment,
    a.gbkbch as first_check_pest,
    a.gbkzw as first_check_impurity,
    a.gbkht as first_insp_reject_blackhead,
    a.gbkbx as first_insp_reject_core_clump,
    a.zckzm as second_check_cane_tail,
    a.zckyq as second_check_cane_leaf,
    a.zckxg as second_check_root,
    a.zckns as second_check_sediment,
    a.zckbch as second_check_pest,
    a.zckzw as second_check_impurity,
    a.zckht as second_insp_reject_blackhead,
    a.zckbx as second_insp_reject_core_clump,
    a.zjy as first_check_inspector,
    zj2.zjy AS second_check_inspector,
    a.Yhqkzl AS pre_discount_imp_cane_qty,
    a.jszjkz AS post_discount_imp_cane_qty,
    a.cdkzl AS over_ton_impurity_ded,

    a.kc AS trash_deduct,
    a.mzs AS arrival_time,          
    jh11.pcrs AS departure_time,
    a.mzs AS all_weight_timestamp,
    a.pzs AS vehicle_weight_timestamp,
    'ALL' AS weight_type,
    ocnt.dump_seq AS dump_seq_no,
    ocnt.dump_code AS dump_aux_code,
    ocnt.dump_code AS dump_number,
    ocnt.tray_code AS tray_code,
    ocnt.create_time AS dump_timestamp,
    ocnt.create_time AS cqms_create_time,
    ocnt.update_time AS cqms_update_time,
    ocnt.create_time AS dump_timestamp2,
    swipe.xzwbh AS queue_code,
    a.zzhn AS receipt_no,
    -- 修正：去掉NVL，与第二个SQL一致
    aj6.jsdmc AS harvest_team,
    aj6.jscxmc AS harvest_vehicle,
    a.zzh AS receipt_no_series,
    NVL(ocnt.ccs, 0) AS ccs,
    NVL(ocnt.raw_ccs, 0) AS raw_ccs,
    NVL(ocnt.Brix, 0) AS Brix,
    NVL(ocnt.Pol, 0) AS Pol,
    NVL(ocnt.Purity, 0) AS Purity,
    NVL(ocnt.Fiber, 0) AS Fiber,
    NVL(ocnt.raw_ccs2, 0) AS raw_ccs2,
    NVL(ocnt.Brix2, 0) AS Brix2,
    NVL(ocnt.Pol2, 0) AS Pol2,
    NVL(ocnt.Purity2, 0) AS Purity2,
    NVL(ocnt.Fiber2, 0) AS Fiber2,
    NVL(ocnt.sucrose_cane, 0) AS sucrose_cane,
    NVL(ocnt.GPj, 0) AS GPj,                          
    NVL(ocnt.RS, 0) AS RS,         
    NVL(ocnt.Bc, 0) AS Bc,         
    0 AS Pc,     
    tc.loading_method,
    tc.kbmc AS loading_method_detail,
    tc.cane_grade_code,     
    tc.cane_grade_group, 
    tc.cane_grade_group_name AS cane_grade_group_name,
    tc.cane_grade_group_name AS cane_grade_name,
    a0.pyjiancheng AS company_code,
    a0.mcjiancheng AS company_name,
    a.factory AS company,
    a0.pyjiancheng AS station_code,
    'COMPANY' AS station_type,
    a.cph AS vehicle_code,
    'FALSE' AS trailer_truck,
    a0.pyjiancheng AS cane_recipient_company_code,
    TRIM(a0.pyjiancheng) || zone.qh AS zone_code,
    zone.qmc AS zone_name,
    a.cph AS vehicle_bp_code,
    a.sjmc AS vehicle_bp_name,
    aj6.jhq AS cut_timestamp,
    '20' || a.season AS be_year,
    a2.xm AS town_code,
    a2.xmc AS town_name,
    a4.cm AS village_code,
    a4.cmc AS village_name,
    a6.dm AS team_code,
    a6.dmc AS team_name,
    a.pz AS variety_code,
    var.pzmc AS variety_name,
    var.flfh AS variety_type_code,
    var_type.flmc AS variety_type_name,
    a.create_time,
    a.update_time,
    a.update_time AS updateDate,
    CASE WHEN a.dczzh = 0 THEN a.zzh ELSE a.dczzh END AS large_vehicle_receipt,
    CASE WHEN NVL(ocnt.dump_code, '99999') = '99999' THEN '0' ELSE '1' END AS is_traceqc,
    NVL(swipe_card.swipe_flag, '0') AS swipe_card,
    swipe.yzx AS press_line,
    a.dw_update_time
FROM 
    -- 修正：主表改为与第二个SQL一致的dwd.dwd_gb_settle_info_i_1h
    dwd.dwd_gb_settle_info_i_1h a
              
LEFT JOIN dwd.dwd_gis_sugarcane_quality_tracking_i_1h ocnt 
    ON  CASE 
    WHEN NVL(CAST(a.dczzh AS INT), 0) > 1 
    THEN SUBSTR(a.zzhn_new, 1, INSTR(a.zzhn_new, '-'))
        || LPAD(CAST(a.dczzh AS STRING), 10, '0')   
    ELSE a.zzhn_new
    END = ocnt.zzhn_new
    AND TRIM(a.factory) = TRIM(ocnt.factory)
LEFT JOIN dwd.dwd_gis_team_info_i_1mon a6 
    ON SUBSTR(a.zzm, 1, 8) = a6.dm
LEFT JOIN dwd.dwd_gis_village_info_i_1mon a4 
    ON a6.cm = a4.cm
LEFT JOIN dwd.dwd_gis_town_info_f_1mon a2 
    ON a4.xm = a2.xm
LEFT JOIN dwd.dwd_gis_sugarcane_owner_info_i_1mon a14 
    ON a.zzm = a14.zzm
LEFT JOIN dwd.dwd_gis_nw_company_info_i_1mon a0 
    ON a.factory = a0.jiagonggc
-- 修正：aj6关联表改为与第二个SQL一致的dwd.dwd_gis_dispatch_plan_info_i_1h
LEFT JOIN dwd.dwd_gis_dispatch_plan_info_i_1h aj6 
    ON a.zzhn = aj6.zzhn
LEFT JOIN dwd.dwd_gis_transfer_station_info_i_1mon zzz 
    ON a.hzzm = zzz.zzzm
LEFT JOIN (
    SELECT 
        a32.cph,
        a32.factory,
        a32.season,
        a31.fl,
        a31.cmc
    FROM dwd.dwd_gis_vehicle_model_info_i_1mon a31
    INNER JOIN dwd.dwd_gis_fleet_info_i_1mon a32 
        ON a31.fl = a32.cx AND a31.factory = a32.factory
    GROUP BY a32.cph, a32.factory, a32.season, a31.fl, a31.cmc               
) vtype 
    ON a.cph = vtype.cph 
    AND aj6.factory = vtype.factory 
    AND a.season = vtype.season
LEFT JOIN dwd.dwd_gis_fleet_info_i_1mon fleet 
    ON a.cph = fleet.cph 
    AND aj6.factory = fleet.factory 
    AND a.season = fleet.season
-- 修正：jh11关联表改为与第二个SQL一致的dwd.dwd_gis_vehicle_dispatch_info_i_1h
LEFT JOIN dwd.dwd_gis_vehicle_dispatch_info_i_1h jh11 
    ON a.zzhn = jh11.zzhn
LEFT JOIN dwd.dwd_gis_second_quality_inspection_f_1h zj2 
    ON  CASE 
    WHEN NVL(CAST(a.dczzh AS INT), 0) > 1 
    THEN SUBSTR(a.zzhn, 1, INSTR(a.zzhn, '-'))
        || LPAD(CAST(a.dczzh AS STRING), 10, '0')
    ELSE a.zzhn
    END = zj2.zzhn
    AND a.season = zj2.season
LEFT JOIN (
    SELECT xzwbh, be_year, factory, zzh,yzx
    FROM (
        SELECT 
            xzwbh,
            be_year,
            factory,
            zzh,
            yzx,
            create_time,
            ROW_NUMBER() OVER (PARTITION BY be_year, factory, zzh ORDER BY create_time DESC) AS rn
        FROM dwd.dwd_cqms_zc_cph_zzh_swipe_info_i_1h
    ) t
    WHERE rn = 1
) swipe 
    ON '20' || a.season = swipe.be_year 
    AND a.factory = swipe.factory 
    AND CASE WHEN a.dczzh = 0 THEN a.zzh ELSE a.dczzh END = swipe.zzh
LEFT JOIN (
    SELECT qh, qmc, dm, factory
    FROM (
        SELECT 
            qh,
            qmc,
            dm,
            factory,
            create_time,
            ROW_NUMBER() OVER (PARTITION BY dm, factory ORDER BY create_time DESC) AS rn
        FROM dwd.dwd_gis_sugarcane_zone_team_i_1mon
    ) t
    WHERE rn = 1
) zone 
    ON SUBSTR(a.zzm, 1, 8) = zone.dm 
LEFT JOIN (
    SELECT 
        kbdm,
        kbmc,
        factory,
        loading_method,
        cane_grade_code,
        cane_grade_group,
        cane_grade_group_name
    FROM (
        SELECT 
            kbdm,
            kbmc,
            factory,
            CASE WHEN fldm = '02' THEN '机收蔗' ELSE '人工砍蔗' END AS loading_method,
            CASE WHEN fldm = '09' OR kbdm = '0202' THEN '2' ELSE '1' END AS cane_grade_code,
            CASE WHEN fldm = '09' OR kbdm = '0202' THEN 'BURN' ELSE 'FRESH' END AS cane_grade_group,
            CASE WHEN fldm = '09' OR kbdm = '0202' THEN '火烧甘蔗' ELSE '新鲜甘蔗' END AS cane_grade_group_name,
            create_time,
            ROW_NUMBER() OVER (PARTITION BY kbdm, factory ORDER BY create_time DESC) AS rn
        FROM dwd.dwd_gis_ticket_category_info_i_1mon
    ) t
    WHERE rn = 1
) tc 
    ON a.pzlb = tc.kbdm 
    AND CASE
    WHEN a.org = a.factory THEN a.org
    WHEN a.factory = '6071' THEN '6011'
    WHEN a.factory = '60412' THEN '6041'
    ELSE a.factory
END = tc.factory
LEFT JOIN dwd.dwd_gis_sugarcane_variety_info_i_1mon var 
    ON a.pz = var.pzm 
    AND TRIM(a.org) = TRIM(var.factory)
LEFT JOIN dwd.dwd_gis_variety_price_category_f_1mon var_type 
    ON var.flfh = var_type.flfh 
    AND var.factory = var_type.factory
LEFT JOIN (
    SELECT 
        a00.zzhn,
        CASE WHEN NVL(oci.cph, '99999') = '99999' THEN '0' ELSE '1' END AS swipe_flag
    FROM dwd.dwd_gb_settle_info_i_1h a00
    LEFT JOIN dwd.dwd_cqms_zc_cph_zzh_swipe_info_i_1h oci 
        ON CASE 
            WHEN NVL(CAST(a00.dczzh AS INT), 0) > 1 
            THEN SUBSTR(a00.zzhn, 1, 12) || SUBSTR(CAST((10000000 + a00.dczzh) AS STRING), 2, 7)
            ELSE a00.zzhn 
           END = oci.zzhn
    GROUP BY a00.zzhn, CASE WHEN NVL(oci.cph, '99999') = '99999' THEN '0' ELSE '1' END
) swipe_card 
    ON a.zzhn = swipe_card.zzhn
WHERE 
    NVL(ocnt.is_repeat, 0) = 0
    AND a0.dt != ''
    {date_cond}
    {season_cond}
ORDER BY 
    a.create_time
    """
    
    df = spark.sql(sql).orderBy(col("create_time").desc()).dropDuplicates(["receipt_no"])
    # 移除耗时的count()打印
    if no_date_filter:
        print(f"\n======= 全量历史数据查询完成 =======")
    else:
        print(f"\n======= {target_month}月数据查询完成 =======")
    return df

def table_exists(spark, database, table_name):
    """检查表是否存在"""
    try:
        spark.sql(f"DESCRIBE {database}.{table_name}")
        return True
    except:
        return False

def get_global_max_id(spark, database, table_name, exclude_dt=None, exclude_dh=None, exclude_dm=None):
    """
    获取全表最大ID（排除指定分区）
    :param spark: SparkSession实例
    :param database: 数据库名
    :param table_name: 表名
    :param exclude_dt: 要排除的dt分区值（如'202601'/'history'）
    :param exclude_dh: 要排除的dh分区值（如'0'/'history'）
    :param exclude_dm: 要排除的dm分区值（如'0'/'history'）
    :return: 排除目标分区后的最大ID，默认返回0
    """
    # 基础查询：全表max_id
    base_sql = f"""
        SELECT COALESCE(MAX(id), 0) AS max_id 
        FROM {database}.{table_name}
    """
    
    # 如果指定了要排除的分区，添加WHERE条件
    if exclude_dt and exclude_dh and exclude_dm:
        base_sql = f"""
            SELECT COALESCE(MAX(id), 0) AS max_id 
            FROM {database}.{table_name}
            WHERE NOT (dt = '{exclude_dt}' AND dh = '{exclude_dh}' AND dm = '{exclude_dm}')
        """
    
    max_id_df = spark.sql(base_sql)
    max_id = max_id_df.collect()[0]['max_id']
    print(f"\n======= 排除分区({exclude_dt}/{exclude_dh}/{exclude_dm})后，全表最大ID：{max_id} =======")
    return int(max_id) if max_id else 0

def process_history_data(spark, target_table):
    """处理历史数据（首次建表，dt=history）"""
    print("\n======================处理历史数据（首次建表）====================")
    # 核心修改：no_date_filter=True → 去掉日期限制，仅筛选season != '25/26'
    df = get_factory_data_by_month(
        spark=spark,
        target_month=None,  # 无需指定月份
        is_history=True,
        no_date_filter=True  # 关键：关闭日期筛选
    ).cache()
    
    # 替换count() == 0为更高效的isEmpty()
    if df.rdd.isEmpty():
        print("无历史数据可处理")
        df.unpersist()
        return
    
    # 添加id和dw_update_time字段
    window = Window.orderBy("create_time")
    final_df = df \
        .withColumn("id", row_number().over(window)) \
        .withColumn("dw_update_time", current_timestamp()) \
        .select(TARGET_FIELD_ORDER)
    
    # 调用writer_las写入history分区
    writer_las(spark, final_df, 'app', target_table, dt='history', dh='history', dm='history', is_overwrite=False)
    print(f"历史数据写入完成")
    df.unpersist()

def process_monthly_data(spark, target_table, target_month):
    """处理单个月度分区数据（2526榨季）"""
    print(f"\n======================处理{target_month}月数据（2526榨季）====================")
    # 获取全表最大ID，保证ID全局唯一
    max_id = get_global_max_id(spark, 'app', target_table)
    
    # 月度数据仍保留日期筛选，no_date_filter默认False
    df = get_factory_data_by_month(
        spark=spark,
        target_month=target_month,
        is_history=False
    ).cache()
    
    # 替换count() == 0为更高效的isEmpty()
    if df.rdd.isEmpty():
        print(f"{target_month}月无数据可处理")
        df.unpersist()
        return
    
    # 添加id（续接全局最大ID）和dw_update_time
    window = Window.orderBy("create_time")
    final_df = df \
        .withColumn("row_num", row_number().over(window)) \
        .withColumn("id", col("row_num") + max_id) \
        .drop("row_num") \
        .withColumn("dw_update_time", current_timestamp()) \
        .select(TARGET_FIELD_ORDER)
    
    # 写入月度分区（dt=yyyyMM，dh/dm=0）
    writer_las(spark, final_df, 'app', target_table, dt=target_month, dh='0', dm='0', is_overwrite=True)
    print(f"{target_month}月数据写入完成")
    df.unpersist()

# 主函数
if __name__ == "__main__":
    print("======================启动Spark会话====================")
    spark = SparkSession.builder \
        .appName("Cane_Quality_Analysis_Monthly_BucketedTable") \
        .enableHiveSupport() \
        .config("spark.sql.sources.bucketing.enabled", "true") \
        .config("spark.sql.adaptive.enabled", "true") \
        .config("spark.sql.adaptive.coalescePartitions.enabled", "true") \
        .config("spark.sql.parquet.enableVectorizedReader", "true") \
        .config("spark.default.parallelism", "200") \
        .config("spark.sql.shuffle.partitions", "200") \
        .getOrCreate()
    
    sc = spark.sparkContext
    sc._conf.set("spark.scheduler.listenerbus.eventqueue.capacity", "10000")
    
    # 配置参数
    DATABASE = "app"
    TARGET_TABLE = "app_cane_quality_analysis_v2_f_1h"
    # 配置需要处理的2526榨季月度列表（可根据实际需求调整）
    """    
    # "202511": 2025-10-29 ~ 2025-11-30
    # "202512": 2025-11-28 ~ 2025-12-31
    # "202601": 2025-12-29 ~ 2026-01-31
    # "202602": 2026-01-29 ~ 2026-02-28
    # "202603": 2026-02-26 ~ 2026-03-31
    # "202604": 2026-03-29 ~ 2026-04-30
    """
    MONTH_LIST = [
#         "202511",
#         "202512",
#         "202601",
#         "202602",
        "202603",
#         "202604",
                 ]
    
    # 执行逻辑
    table_exists_flag = table_exists(spark, DATABASE, TARGET_TABLE)
    print(f"\n目标表 {DATABASE}.{TARGET_TABLE} 是否存在：{table_exists_flag}")
    
    if not table_exists_flag:
        # 首次建表：先处理全量历史数据（仅season != '25/26'，无日期限制）
        process_history_data(spark, TARGET_TABLE)
    else:
        # 表已存在：仅增量处理2526榨季月度数据
        for month in MONTH_LIST:
            process_monthly_data(spark, TARGET_TABLE, month)
    
    print("\n======================关闭Spark会话====================")
    spark.stop()

======================启动Spark会话====================

目标表 app.app_cane_quality_analysis_v2_f_1h 是否存在：True

======================处理202603月数据（2526榨季）====================

======= 排除分区(None/None/None)后，全表最大ID：2044811364 =======

======= 开始刷新202603月依赖表 =======
✅ 成功刷新表：dwd.dwd_gb_settle_info_i_1h
✅ 成功刷新表：dwd.dwd_gis_sugarcane_quality_tracking_i_1h
✅ 成功刷新表：dwd.dwd_gis_team_info_i_1mon
✅ 成功刷新表：dwd.dwd_gis_village_info_i_1mon
✅ 成功刷新表：dwd.dwd_gis_town_info_f_1mon
✅ 成功刷新表：dwd.dwd_gis_sugarcane_owner_info_i_1mon
✅ 成功刷新表：dwd.dwd_gis_nw_company_info_i_1mon
✅ 成功刷新表：dwd.dwd_gis_dispatch_plan_info
✅ 成功刷新表：dwd.dwd_gis_transfer_station_info_i_1mon
✅ 成功刷新表：dwd.dwd_gis_vehicle_model_info_i_1mon
✅ 成功刷新表：dwd.dwd_gis_fleet_info_i_1mon
✅ 成功刷新表：dwd.dwd_gis_vehicle_dispatch_info
✅ 成功刷新表：dwd.dwd_gis_second_quality_inspection_f_1h
✅ 成功刷新表：dwd.dwd_cqms_zc_cph_zzh_swipe_info_i_1h
✅ 成功刷新表：dwd.dwd_gis_sugarcane_zone_team_i_1mon
✅ 成功刷新表：dwd.dwd_gis_ticket_category_info_i_1mon
✅ 成功刷新表：dwd.dwd_gis_sugarcane_variety_info